# Numerical Methods and Their Accuracy

Every ODE solver advances in small steps using slopes sampled around the current point. The number of slopes per step, and how they are combined, sets the **order**: the power of $h$ in the error. This notebook codes Euler, Heun and classical Runge-Kutta, measures their orders, shows what goes wrong when the step is too large, and ends with adaptive step-size control.

In [ ]:
Pollis.packages("OrdinaryDiffEq", "Plots")

In [ ]:
using OrdinaryDiffEq, Plots
f(y, t) = y * cos(t)
exact(t) = exp(sin(t))   # the solution with y(0) = 1

## Three one-step methods

- **Euler:** one slope, at the start.
- **Heun:** the average of the slope at the start and at the end of an Euler step.
- **RK4:** four slopes (start, two at the midpoint, end) with weights $1, 2, 2, 1$ over 6, like Simpson's rule.

In [ ]:
euler_step(f, y, t, h) = y + h * f(y, t)
heun_step(f, y, t, h) = (k1 = f(y, t); k2 = f(y + h * k1, t + h); y + h / 2 * (k1 + k2))
function rk4_step(f, y, t, h)
    k1 = f(y, t)
    k2 = f(y + h / 2 * k1, t + h / 2)
    k3 = f(y + h / 2 * k2, t + h / 2)
    k4 = f(y + h * k3, t + h)
    return y + h / 6 * (k1 + 2k2 + 2k3 + k4)
end
function march(step, f, y0, T, n)
    h = T / n
    y = y0
    for i in 0:n-1
        y = step(f, y, i * h, h)
    end
    return y
end

## Measuring the order

On a log-log plot of error against step size, a method of order $p$ gives a line of slope $p$. The observed order is $\log_2$ of the ratio of successive errors when $h$ halves.

In [ ]:
T = 10.0
ns = [10 * 2^k for k in 0:7]
schemes = ["Euler" => euler_step, "Heun" => heun_step, "RK4" => rk4_step]
errs = [[abs(march(s, f, 1.0, T, n) - exact(T)) for n in ns] for (_, s) in schemes]
plot(T ./ ns, errs, xscale=:log10, yscale=:log10, m=:circle, lw=2, label=permutedims(first.(schemes)), xlabel="step h", ylabel="error at t = 10", legend=:bottomright)

In [ ]:
[(method = first(schemes[i]), observed_order = log2(errs[i][4] / errs[i][5])) for i in eachindex(schemes)]

## Cost against accuracy

RK4 costs four evaluations of $f$ per step against Euler's one, but the extra work pays: for any accuracy beyond two digits it needs far fewer evaluations in total.

In [ ]:
evals = [1, 2, 4]   # evaluations of f per step
plot([evals[i] .* ns for i in 1:3], errs, xscale=:log10, yscale=:log10, m=:circle, lw=2, label=permutedims(first.(schemes)), xlabel="evaluations of f", ylabel="error at t = 10", title="Work against precision")

## Stability: when the step is too large

For $y' = -50(y - \cos t)$ the solution is pulled towards $\cos t$ with rate 50. Euler multiplies each deviation by $1 - 50h$ per step. That factor exceeds 1 in size once $h > 2/50 = 0.04$, and the numerical solution then explodes even though the true solution is perfectly tame.

In [ ]:
g(y, t) = -50 * (y - cos(t))
p = plot(xlabel="t", ylabel="y", title="Euler on a stiff equation", ylim=(-2, 2))
for h in (0.03, 0.045)
    ts = 0:h:1.5
    ys = accumulate((y, t) -> euler_step(g, y, t, h), ts[1:end-1]; init=0.0)
    plot!(p, ts, [0.0; ys], m=:circle, ms=2, label="h = $h, 50h = $(50h)")
end
p

## Adaptive step-size control

Tsit5 computes a fifth-order and a fourth-order answer from the same six slopes; their difference estimates the error of the step. Steps whose estimate exceeds the tolerance are rejected and retried smaller. Tightening the tolerance by 1000 costs only a few times more work.

In [ ]:
prob = ODEProblem((y, p, t) -> y * cos(t), 1.0, (0.0, 10.0))
map((1e-3, 1e-6, 1e-9, 1e-12)) do tol
    s = solve(prob, Tsit5(); reltol=tol, abstol=tol / 1000)
    (reltol = tol, steps = length(s.t) - 1, rejected = s.stats.nreject, f_evaluations = s.stats.nf, error = abs(s.u[end] - exact(10.0)))
end

On the stiff equation, the adaptive explicit solver is safe, since it never explodes, but it is slow: its steps stay near its stability limit. The implicit Rodas5P lengthens its steps as soon as the fast transient is over. That is the subject of notebook 5.

In [ ]:
stiffprob = ODEProblem((y, p, t) -> -50 * (y - cos(t)), 0.0, (0.0, 10.0))
se = solve(stiffprob, Tsit5(); reltol=1e-4); si = solve(stiffprob, Rodas5P(); reltol=1e-4)
plot(se.t[2:end], diff(se.t), yscale=:log10, m=:circle, ms=2, label="Tsit5: $(length(se.t) - 1) steps", xlabel="t", ylabel="step size")
plot!(si.t[2:end], diff(si.t), m=:square, ms=2, label="Rodas5P: $(length(si.t) - 1) steps")

## Exercises

1. Add the midpoint method, $y_{n+1} = y_n + h f(y_n + \tfrac h2 f(y_n, t_n), t_n + \tfrac h2)$, and measure its order.
2. Where exactly does Euler's error for $y' = -50(y - \cos t)$ switch from bounded to growing? Try $h$ = 0.039 and 0.041.
3. Heun's method is stable for $|1 - z + z^2/2| \le 1$ with $z = 50h$. What is its step limit on the stiff equation?
4. Why does RK4's observed order fall away from 4 at the smallest steps in the plot?